In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
n_items, n_dim, n_classses, k = 100, 2, 2, 5

current_vectorised_items = np.random.rand(n_items, n_dim).tolist()
labels = np.random.randint(0, n_classses, size=n_items).tolist()
queries = np.random.rand(10, n_dim).tolist()

## Naive

In [ ]:
assignments = []

# Trivial parallelisation possible over queries, ignore that
for q in queries:
    
    # Now, we want to compute KNN for a given query against N items
    ## We compute O(N) euclidean distances
    dist_and_indices = []
    for i, x in enumerate(current_vectorised_items):
        _dist = sum((a-b)**2 for a,b in zip(q,x))**0.5
        dist_and_indices.append((i, _dist))
    
    # Then sort, roughly O(n.logn) average case e.g. mergesort
    dist_and_indices = sorted(dist_and_indices, key=lambda x: x[1])
    
    # Then, get class labels, ignore this loop as trivial O(k)
    top_k = dist_and_indices[:k]
    counts = {}
    for cls in [labels[i] for i,_ in top_k]:
        counts[cls] = counts.get(cls, 0) + 1
    max_count = max(counts.values())
    majority = [cls for cls, cnt in counts.items() if cnt == max_count][-1]

    assignments.append(majority)
    
# Overall case
## We do O(N) + O(n.logn) ~ O(n.logn)

## Can use helper functions e.g. euclidean
## Can drop square root as constant factor

In [ ]:
plt.figure(figsize=(8,6))
plt.scatter([x[0] for x in current_vectorised_items], [x[1] for x in current_vectorised_items], c=labels, label='Data', s=60)
plt.scatter([x[0] for x in queries], [x[1] for x in queries], c=assignments, marker="D", s=120, label='Queries')
plt.legend();

## Advanced (via minheap)

In [ ]:
class KHeap:
    def __init__(self, k):
        self.heap = []  # list of (index, distance)
        self.k = k

    def insert(self, item):
        """Insert an item (index, distance) into the max-heap"""
        if len(self.heap) < self.k:
            # Append at end and sift-up
            self.heap.append(item)
            self._sift_up(len(self.heap) - 1)
        else:
            # Compare with max (root)
            if item[1] < self.heap[0][1]:
                # Replace root and sift-down
                self.heap[0] = item
                self._sift_down(0)

    def _sift_up(self, idx):
        """Move the element at idx up to restore max-heap"""
        while idx > 0:
            parent = (idx - 1) // 2
            if self.heap[idx][1] > self.heap[parent][1]:
                self.heap[idx], self.heap[parent] = self.heap[parent], self.heap[idx]
                idx = parent
            else:
                break

    def _sift_down(self, idx):
        """Move the element at idx down to restore max-heap"""
        n = len(self.heap)
        while True:
            largest = idx
            left = 2 * idx + 1
            right = 2 * idx + 2

            if left < n and self.heap[left][1] > self.heap[largest][1]:
                largest = left
            if right < n and self.heap[right][1] > self.heap[largest][1]:
                largest = right

            if largest != idx:
                self.heap[idx], self.heap[largest] = self.heap[largest], self.heap[idx]
                idx = largest
            else:
                break

def pseudo_euclidean(q,x):
    return sum((a-b)**2 for a,b in zip(q,x))

In [ ]:
## Leveraging minheap data structure, we don't need to do the O(n.logn) sorting, but keep perfect retrieval
assignments = []

# Trivial parallelisation possible over queries, ignore that
for q in queries:
    
    # Now, we want to compute KNN for a given query against N items
    ## We compute O(N) euclidean distances
    heap = KHeap(k=k)
    for i, x in enumerate(current_vectorised_items):
        _dist = pseudo_euclidean(q,x) # skipped square root
        heap.insert((i, _dist))
    
    # Then, get class labels, ignore this loop as trivial O(k)
    counts = {}
    for cls in [labels[i] for i,_ in heap.heap]:
        counts[cls] = counts.get(cls, 0) + 1
    max_count = max(counts.values())
    majority = [cls for cls, cnt in counts.items() if cnt == max_count][-1]

    assignments.append(majority)
    
# Overall case
## We do O(N) + O(n.logk) ~ O(n.logk)

In [ ]:
plt.figure(figsize=(8,6))
plt.scatter([x[0] for x in current_vectorised_items], [x[1] for x in current_vectorised_items], c=labels, label='Data', s=60)
plt.scatter([x[0] for x in queries], [x[1] for x in queries], c=assignments, marker="D", s=120, label='Queries')
plt.legend();

## Advanced (approximate via KD Tree)

In [ ]:
class KDTree:
    def __init__(self, points, depth=0, indices=None):
        self.axis = None
        self.point = None
        self.index = None
        self.left = None
        self.right = None
        self.depth = depth

        if not points:
            return

        self.n_dims = len(points[0])
        if indices is None:
            indices = list(range(len(points)))

        axis = depth % self.n_dims
        sorted_data = sorted(zip(points, indices), key=lambda x: x[0][axis])
        median_idx = len(sorted_data) // 2
        self.point, self.index = sorted_data[median_idx]

        left_data = sorted_data[:median_idx]
        right_data = sorted_data[median_idx + 1:]

        if left_data:
            self.left = KDTree([p for p, _ in left_data], depth + 1, [i for _, i in left_data])
        if right_data:
            self.right = KDTree([p for p, _ in right_data], depth + 1, [i for _, i in right_data])

    def descend_to_leaf(self, query):
        """Go down to the leaf where the query falls."""
        node = self
        while node.left or node.right:
            axis = node.depth % len(query)
            if query[axis] < node.point[axis] and node.left:
                node = node.left
            elif node.right:
                node = node.right
            else:
                break
        return node

    def collect_leaf_points(self):
        """Collect all points in this subtree (used for approximation)."""
        result = []
        def dfs(node):
            if not node:
                return
            result.append((node.index, node.point))
            dfs(node.left)
            dfs(node.right)
        dfs(self)
        return result


In [ ]:
assignments = []
tree = KDTree(current_vectorised_items)

for q in queries:
    # Step 1: descend to a leaf
    leaf = tree.descend_to_leaf(q)

    # Step 2: collect nearby points (could include siblings if you want)
    candidates = leaf.collect_leaf_points()

    # Step 3: take last K or all if fewer
    top_k = [(i, np.linalg.norm(np.array(p) - np.array(q))) for i, p in candidates]
    top_k = sorted(top_k, key=lambda x: x[1])[:k]

    # Step 4: voting
    counts = {}
    for cls in [labels[i] for i, _ in top_k]:
        counts[cls] = counts.get(cls, 0) + 1
    max_count = max(counts.values())
    majority = [cls for cls, cnt in counts.items() if cnt == max_count][-1]

    assignments.append(majority)


In [ ]:
plt.figure(figsize=(8,6))
plt.scatter([x[0] for x in current_vectorised_items], [x[1] for x in current_vectorised_items], c=labels, label='Data', s=60)
plt.scatter([x[0] for x in queries], [x[1] for x in queries], c=assignments, marker="D", s=120, label='Queries')
plt.legend();